# Rotulagem de conservação dos imóveis

Gera `dados/rotulos_conservacao.csv` para treinar o classificador próprio (Fase 2).

## Labels (olhe só o visível na foto)
- **bem conservado**: pintura/piso/banheiro aparentemente novos, sem marcas
- **desgaste leve**: marcas leves de uso, tudo funcional
- **desgastado**: desgaste evidente (pintura gasta, piso manchado, rejunte escuro)
- **precário**: mofo, infiltração, danos ou risco aparente

## Comandos (digite + Enter)
- `1-4` rotula | `0` pula | `z` desfaz o último | `q` salva e sai
- Salvamento incremental: nada se perde se interromper; rode de novo que pula as já rotuladas
- Pode re-rodar a célula do loop sem a célula 2: fotos já rotuladas são puladas sozinhas

**Meta**: ≥80 fotos por classe (total ~400).

Filtros da amostra: só `apartamento`/`casa`, sem fotos placeholder (`nao-encontrado`).

In [7]:
from pathlib import Path
import random
import pandas as pd

CIDADE = "joinville"
N = 400
SEED = 42
LABELS = ["bem conservado", "desgaste leve", "desgastado", "precário"]

REPO = Path.cwd()
if not (REPO / "dados").exists():
    REPO = REPO.parent  # notebook aberto a partir de Notebooks/
CSV_PATH = REPO / "dados" / "rotulos_conservacao.csv"

parquets = sorted((REPO / "dados" / CIDADE).glob(f"{CIDADE}_imoveis_limpo_*.parquet"))
assert parquets, f"nenhum parquet em dados/{CIDADE}"
df = pd.read_parquet(parquets[-1], columns=["url", "titulo", "bairro", "valor_imovel", "tipo_imovel", "fotos"])
print("parquet:", parquets[-1].name, "| linhas:", len(df))
df = df[df["fotos"].notna()].copy()
df = df[df["fotos"].map(lambda f: len(f) > 0)]
print("com fotos:", len(df))
# só apartamentos e casas
df = df[df["tipo_imovel"].isin(["apartamento", "casa"])].copy()
print("apto/casa:", len(df))
# fora placeholders (ex.: chavesnamao 'nao-encontrado'); mantém anúncios com ≥1 foto real
df["fotos"] = df["fotos"].map(lambda fs: [u for u in list(fs) if "nao-encontrado" not in str(u)])
df = df[df["fotos"].map(lambda f: len(f) > 0)].copy()
print("sem placeholder:", len(df))

rng = random.Random(SEED)
# 1 foto aleatória por anúncio (diversifica cômodos)
df["foto"] = df["fotos"].map(lambda f: rng.choice(list(f)))

# estratificada por bairro (proporcional, mín 1 por bairro)
quotas = (df["bairro"].value_counts(normalize=True) * N).round().astype(int)
quotas[quotas < 1] = 1
partes = []
for bairro, qtd in quotas.items():
    g = df[df["bairro"] == bairro]
    partes.append(g.sample(min(len(g), qtd), random_state=SEED))
amostra = pd.concat(partes).sample(frac=1, random_state=SEED).head(N).reset_index(drop=True)

# resume: remove já rotuladas
if CSV_PATH.exists():
    feitas = set(pd.read_csv(CSV_PATH)["url_foto"].tolist())
    amostra = amostra[~amostra["foto"].isin(feitas)].reset_index(drop=True)
    print(f"resume: {len(feitas)} já rotuladas puladas")
print("a rotular:", len(amostra))
amostra[["bairro", "titulo", "valor_imovel"]].head(3)

parquet: joinville_imoveis_limpo_2026-10.parquet | linhas: 26912
com fotos: 26866
apto/casa: 23805
sem placeholder: 19301
resume: 141 já rotuladas puladas
a rotular: 400


,bairro,titulo,valor_imovel
0,anita garibaldi,"Cobertura com 3 Quartos à venda, 166m² - Anita...",1600000.0
1,atiradores,Apartamento com 3 quartos à venda na Rua Aquid...,3019429.0
2,jarivatuba,Apartamento 2 quartos Paranaguamirim Joinville SC,250000.0


In [8]:
amostra

,url,titulo,bairro,valor_imovel,tipo_imovel,fotos,foto
0,https://www.zapimoveis.com.br/imovel/venda-cob...,"Cobertura com 3 Quartos à venda, 166m² - Anita...",anita garibaldi,1600000.0,apartamento,[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://resizedimgs.zapimoveis.com.br/img/vr-l...
1,https://www.chavesnamao.com.br/imovel/apartame...,Apartamento com 3 quartos à venda na Rua Aquid...,atiradores,3019429.0,apartamento,[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.chavesnamao.com.br/imn/1200x0800/N...
2,https://sc.olx.com.br/norte-de-santa-catarina/...,Apartamento 2 quartos Paranaguamirim Joinville SC,jarivatuba,250000.0,apartamento,[https://img.olx.com.br/images/21/219635442807...,https://img.olx.com.br/images/21/2166562014107...
3,https://www.chavesnamao.com.br/imovel/apartame...,Apartamento com 4 quartos à venda na Rua Visco...,america,3083180.0,apartamento,[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.chavesnamao.com.br/imn/1200x0800/N...
4,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 3 Quartos à venda, 82m² - América",america,799000.0,apartamento,[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://resizedimgs.zapimoveis.com.br/img/vr-l...
...,...,...,...,...,...,...,...
395,https://www.vivareal.com.br/imovel/casa-4-quar...,"Casa com 4 Quartos à venda, 170m² - Saguaçu",saguacu,890000.0,casa,[https://resizedimgs.vivareal.com/img/vr-listi...,https://resizedimgs.vivareal.com/img/vr-listin...
396,https://sc.olx.com.br/norte-de-santa-catarina/...,Casa Averbada 2 Suíte Profipo Joinville,area rural,510000.0,casa,[https://img.olx.com.br/images/97/979646791015...,https://img.olx.com.br/images/95/9506147950513...
397,https://www.imovelweb.com.br/propriedades/gemi...,Geminado no Bairro América,america,1490000.0,casa,[https://imgbr.imovelwebcdn.com/avisos/2/30/03...,https://imgbr.imovelwebcdn.com/avisos/2/30/03/...
398,https://sc.olx.com.br/norte-de-santa-catarina/...,CASA RESIDENCIAL em Joinville - Santa Catarina...,floresta,750000.0,casa,[https://img.olx.com.br/images/20/205613206995...,https://img.olx.com.br/images/20/2016442071298...


In [ ]:
import csv
from datetime import datetime
from IPython.display import Image, display

TECLAS = {"1": LABELS[0], "2": LABELS[1], "3": LABELS[2], "4": LABELS[3]}

COLUNAS = ["url_foto", "url_anuncio", "bairro", "titulo", "valor_imovel", "label", "timestamp"]


def _carregar_existentes():
    if not CSV_PATH.exists():
        return []
    with open(CSV_PATH, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _reescrever(rotulos):
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS)
        w.writeheader()
        w.writerows(rotulos)


rotulos = _carregar_existentes()
base_len = len(rotulos)
vistos = {r["url_foto"] for r in rotulos}  # anti-duplicata: re-rodar o loop sem a célula 2 não repete fotos
salvos = 0
for k, row in amostra.iterrows():
    if row["foto"] in vistos:
        continue
    display(Image(url=row["foto"], width=650))
    print(f"[{k + 1}/{len(amostra)}] {row['bairro']} | {str(row['titulo'])[:70]} | {row['valor_imovel']}")
    while True:
        t = input("1=bem conservado 2=desgaste leve 3=desgastado 4=precário | 0 pula | z desfaz | q sai: ").strip().lower()
        if t in TECLAS:
            rotulos.append({
                "url_foto": row["foto"], "url_anuncio": row["url"],
                "bairro": row["bairro"], "titulo": row["titulo"],
                "valor_imovel": row["valor_imovel"], "label": TECLAS[t],
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            _reescrever(rotulos)
            salvos += 1
            vistos.add(row["foto"])
            break
        if t == "0":
            break
        if t == "z":
            if len(rotulos) > base_len:
                rotulos.pop()
                salvos -= 1
                _reescrever(rotulos)
                print("último desfeito")
            else:
                print("nada a desfazer nesta sessão")
            continue
        if t == "q":
            break
        print("tecla inválida")
    if t == "q":
        print(f"encerrado: +{salvos} rótulos nesta sessão (total {len(rotulos)})")
        break
else:
    print(f"amostra concluída: +{salvos} rótulos (total {len(rotulos)})")

[3/400] jarivatuba | Apartamento 2 quartos Paranaguamirim Joinville SC | 250000.0


[4/400] america | Apartamento com 4 quartos à venda na Rua Visconde de Mauá, 1515, Améri | 3083180.0


[5/400] america | Apartamento com 3 Quartos à venda, 82m² - América | 799000.0


[6/400] anita garibaldi | APARTAMENTO no ANITA GARIBALDI com 3 quartos para VENDA, 173 m² | 1516590.0


[7/400] saguacu | Apartamento com 3 quartos à venda na Rua Aracaju, 179, Saguaçu, Joinvi | 1499179.0


...

[28/400] santo antonio | Apartamento com 2 Quartos à venda, 65m² - Santo Antônio | 429000.0


[29/400] floresta | Casa com 2 Quartos à venda, 67m² - Floresta | 420000.0


[30/400] america | Casa com 4 Quartos à venda, 159m² - América | 890000.0


[31/400] saguacu | Apartamento com 3 quartos à venda na Rua Dona Francisca, 2403, Saguaçu | 1125442.0


[32/400] costa e silva | Apartamento no Costa e Silva em Condomínio Club com Vaga Coberta | 396000.0
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida
tecla inválida


In [6]:
rotulos

[{'url_foto': 'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/40914421/sc-joinville-anita-garibaldi-rua-corupa-apartamento-a-venda-2-quartos-69c7cf6f-3.jpg',
  'url_anuncio': 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-anita-garibaldi-101m2-RS780000/id-40914421/',
  'bairro': 'anita garibaldi',
  'titulo': 'Apartamento com 2 quartos à venda na Rua Corupá, 416, Anita Garibaldi, Joinville',
  'valor_imovel': '780000.0',
  'label': 'bem conservado',
  'timestamp': '2026-10-09T14:05:44'},
 {'url_foto': 'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/323534/28973442/sc-joinville-iririu-rua-arco-iris-casa-sobrado-a-venda-3-quartos-6a9d73de-14.jpg',
  'url_anuncio': 'https://www.chavesnamao.com.br/imovel/casa-a-venda-3-quartos-com-garagem-sc-joinville-iririu-113m2-RS519000/id-28973442/',
  'bairro': 'iririu',
  'titulo': 'Casa com 3 quartos à venda na Rua Arco-Íris, 25, Iririú, Joinville',
  'valor_imovel': '519000

In [ ]:
import pandas as pd

if CSV_PATH.exists():
    rel = pd.read_csv(CSV_PATH)
    print("total:", len(rel))
    cont = rel["label"].value_counts()
    print(cont.to_string())
    print("\nfalta p/ meta de 80/classe:")
    for lab in LABELS:
        print(f"  {lab}: {max(0, 80 - cont.get(lab, 0))}")
    rel.head(3)
else:
    print("nada rotulado ainda — rode a célula do loop")